In [3]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import os


Define parameters


In [68]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io
import sys


def get_data_noaa(lat, lon, year):
    #  The time zone used by Meteostat is Coordinated Universal Time (UTC).
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year-1, 12, 31)
    end = datetime(year+1, 1, 2)

    # Use certifi's CA bundle
    # ssl_context = ssl.create_default_context(cafile=certifi.where())
    # Find the closest station
    stations = Stations().nearby(lat, lon) 
    # Get hourly data for the first station
    data = Hourly(stations.fetch(1), start, end, model=True).fetch()
    timezone = stations.fetch(1)['timezone'].values[0]
    #elevation in meters
    elevation = stations.fetch(1)['elevation'].values[0]
    #Distance in m
    distance = stations.fetch()['distance'].values[0]
    print('distance')
    print(distance)
    #WMO
    wmo = str(stations.fetch(1).index.values[0])
    #Station Name
    station_name = stations.fetch()['name'].values[0]
    #State
    state = stations.fetch()['region'].values[0]
    #Country
    country = stations.fetch()['country'].values[0]

    return data, timezone, distance, elevation, wmo, station_name, state, country

def convert_utc_to_local(df, local_tz):
    """
    Convert the datetime index of the DataFrame from UTC to a local timezone.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index in UTC.
    local_tz : str
        A timezone string (e.g., 'America/Chicago').

    Returns:
    pandas.DataFrame
        DataFrame with datetime index converted to the specified local timezone.
    """

    # Check if the index is in datetime format
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        # Reset the index to remove the multi-index structure
        df = df.reset_index(level='station', drop=True)
        # Convert the 'time' index to a datetime format
        df.index = pd.to_datetime(df.index)

    # Ensure the index is timezone-aware, set to UTC
    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    # Convert the timezone of the index
    df.index = df.index.tz_convert(local_tz)

    return df

def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    """
    Filter the DataFrame to include rows between the specified start and end dates,
    handling timezone differences appropriately.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index.
    start_date : str or datetime-like
        The beginning date of the interval to filter the DataFrame.
    end_date : str or datetime-like
        The end date of the interval to filter the DataFrame.
    timezone : str, optional
        The timezone to which to convert the dates before filtering,
        if the datetime index is timezone-aware.

    Returns:
    pandas.DataFrame
        DataFrame filtered to include only rows between the specified dates.
    """
    # Convert string dates to datetime, considering the timezone
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    
    if timezone:
        # Convert dates to the specified timezone if provided
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        # Make datetime index timezone-naive if no timezone is specified
        df.index = df.index.tz_localize(None)

    # Filter the DataFrame
    mask = (df.index >= start_date) & (df.index <= end_date)
    return df.loc[mask]

def get_parameters_MERRA2(lat, lon, year):
    api_endpoint = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}1231&format=EPW"
    response = requests.get(api_endpoint)
    # Split the response text into lines
    lines = response.text.splitlines()
    header = ('\n'.join(lines[:8]))
    # Convert back into a file-like object
    csv_data = io.StringIO('\n'.join(lines))
    # Read into a pandas DataFrame
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    return df, header

def merge_data(df, data):

    if not df[6].isna().all():
        df[6] = list(data['temp'][1:])
    if not df[7].isna().all():
        df[7] = list(data['temp'][1:])
    if not df[8].isna().all():
        df[8] = list(data['temp'][1:])
    if not df[33].isna().all():
        df[33] = list(data['temp'][1:])
    if not df[30].isna().all():
        df[30] = list(data['temp'][1:])
    if not df[21].isna().all():
        df[21] = list(data['temp'][1:])
    if not df[20].isna().all():
        df[20] = list(data['temp'][1:])
    if not df[9].isna().all():
        df[9] = list(data['temp'][1:])

    return df

def check_missing_hours(year, df):
    # Generate a complete set of hourly timestamps for the entire year
    full_index = pd.date_range(start=f"{year}-01-01 00:00:00", end=f"{year+1}-01-01 00:00:00", freq="h")
    # Find the missing hours by comparing the complete set with the dataframe's index
    missing_hours = full_index.difference(df.index)
    # Calculate the number of missing hours
    missing_hours_num = len(missing_hours)
    # Find the size of the largest group of consecutive missing hours
    if missing_hours_num > 0:
        # Calculate the difference between consecutive missing hours
        diffs = missing_hours.to_series().diff().dt.total_seconds().div(3600)
        # Identify the groups where the difference between consecutive hours is 1 (consecutive hours)
        consecutive_groups = (diffs != 1).cumsum()
        # Find the size of the largest group of consecutive missing hours
        largest_consecutive_group = consecutive_groups.value_counts().max()
    else:
        largest_consecutive_group = 0
    return missing_hours_num, largest_consecutive_group

def get_noaa_merra2_data(lat, lon, year,file_type):
    retrieve_status = True
    # The time zone used by Meteostat is Coordinated Universal Time (UTC).
    data_noaa, tz, distance, elevation, wmo, station_name, state, country = get_data_noaa(lat, lon, year)
    #If Meteostat returns not a WMO, check the correspondoing WMO from NOAA
    if wmo is not None:
        try:
            wmo = str(int(wmo))
        except ValueError:
            # If conversion fails, assume wmo is actually an ICAO code and try to get WMO
            icao = wmo
            wmo = get_wmo_from_icao_NOAA(icao) or icao
        
    
    print(wmo)
    
        
    # Create the dictionary
    info_dict = {
        'timeshift': get_time_shift(tz),
        'elevation': elevation,
        'wmo': wmo,
        'station_name': station_name,
        'state': state,
        'country': country,
        'lat': lat,
        'lon': lon,
        'weather_file_type': file_type
        }
    
    # Adjust timezone and cut from 1/1 to 12/31
    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        #we don't have NOAA data for this location/year
        print("We don't have NOAA data for this location/year")
        retrieve_status = False
        df_merged = []
        distance = np.nan
        cdd = ''
        hdd = ''

    else:
        #Check if there are missing hours:
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data_noaa_tz_adj)
        if largest_consecutive_group>3:
            print('More than 3 consecutive missing hours')
            df_merged = []
            retrieve_status = False
            distance = np.nan
            cdd = ''
            hdd = ''

        else:
            # Resample the dataframe to hourly frequency
            data_noaa_tz_adj_h = data_noaa_tz_adj.resample('h').mean()
            # Linearly interpolate missing values, limiting to 3 consecutive missing hours
            data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='forward')
            hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
            #get data MERRA2
            df_merra2, header_merra2 = get_parameters_MERRA2(lat, lon, year)
            #Merge two datasets
            df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    return df_merged, retrieve_status, info_dict, distance, hdd, cdd

def get_dst_start_end(year, latitude, longitude):
    # Get the timezone for the given latitude and longitude
    tf = TimezoneFinder()
    timezone_str = tf.timezone_at(lat=latitude, lng=longitude)
    
    if timezone_str is None:
        raise ValueError("Could not find timezone for the given coordinates.")
    
    # Get the timezone object
    timezone = pytz.timezone(timezone_str)
    
    # Define the dates for the beginning and end of the year (naive datetime)
    start_of_year = datetime(year, 1, 1)
    end_of_year = datetime(year, 12, 31)
    
    dst_start = None
    dst_end = None

    # Start by localizing the first date
    previous_offset = timezone.localize(start_of_year).dst()

    # Loop through each day of the year
    for dt in [start_of_year + timedelta(days=i) for i in range((end_of_year - start_of_year).days + 1)]:
        localized_dt = timezone.localize(dt)  # Localize naive datetime
        current_offset = localized_dt.dst()
        
        if previous_offset == timedelta(0) and current_offset != timedelta(0):
            dst_start = localized_dt
        elif previous_offset != timedelta(0) and current_offset == timedelta(0):
            dst_end = localized_dt
            break
        
        previous_offset = current_offset
    
    return dst_start, dst_end

def find_closest_design_condition(lat,lon,design_conditions_file):
    """
    Finds the closest design condition from the CSV file based on the given latitude and longitude.

    Parameters:
    lat (float): The latitude of the target location.
    lon (float): The longitude of the target location.
    csv_file (str): The path to the CSV file containing design conditions.

    Returns:
    str: The 2021 design condition string for the closest location.
    """

    # Function to calculate the distance between two points given their latitudes and longitudes
    def haversine_distance(lat1, lon1, lat2, lon2):
        # Convert latitude and longitude from degrees to radians
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        
        # Haversine formula
        dlat = lat2 - lat1 
        dlon = lon2 - lon1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        r = 6371  # Radius of Earth in kilometers. Use 3956 for miles. Determines return value units.
        return c * r

    
    # Read the CSV file
    df = pd.read_csv(design_conditions_file)

    # Calculate distance from target coordinates to each row in the dataframe
    df['distance'] = df.apply(lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']), axis=1)

    # Find the row with the minimum distance
    closest_row = df.loc[df['distance'].idxmin()]

    # Return the design conditions for 2021
    return closest_row['2021_design_conditions']

def create_header(df, year, info_dict):
    header_lines = []

    #Calculated parameters 
    first_day_year = pd.to_datetime(date.min.replace(year=year)).day_name()
    leap_status = lambda year: 'Yes' if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 'No'
    dst_start, dst_end = get_dst_start_end(year, info_dict['lat'], info_dict['lon'])
    design_conditions_file = 'resources/design_conditions.csv'
    design_conditions_line = find_closest_design_condition(float(info_dict['lat']), float(info_dict['lon']), design_conditions_file)
    #Hardcoded parameters
    number_of_holidays = 0
    number_of_data_periods = 1
    number_of_records_per_hour = 1

    # line_1
    header_lines.append(f"LOCATION,{info_dict['station_name']},{info_dict['state']},{info_dict['country']},{info_dict['weather_file_type']},{info_dict['wmo']},{info_dict['lat']},{info_dict['lon']},{info_dict['timeshift']},{info_dict['elevation']}")
    # line_2
    header_lines.append(design_conditions_line)
    # line_3
    header_lines.append(f"TYPICAL/EXTREME PERIODS,0")
    # line_4
    header_lines.append(f"GROUND TEMPERATURES,0")
    # header_lines.append(f"GROUND TEMPERATURES,3,.5,,,,-16.34,-17.80,-15.22,-11.16,-0.57,7.61,13.13,14.81,11.95,5.60,-2.89,-10.76,2,,,,-10.97,-13.57,-13.04,-10.89,-3.80,2.61,7.74,10.49,9.90,6.30,0.46,-5.74,4,,,,-6.53,-9.19,-9.78,-8.97,-4.96,-0.64,3.32,6.08,6.72,5.16,1.73,-2.4")
    # line_5
    try:
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},{dst_start.month}/{dst_start.day},{dst_end.month}/{dst_end.day},{number_of_holidays}")
    except AttributeError:
        #We cannot retrieve DST dates, let's set them to 0
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},0,0,{number_of_holidays}")
    # line_6
    header_lines.append(f"COMMENTS 1, ")
    # line_7
    header_lines.append(f"COMMENTS 2, ")
    # line_8
    header_lines.append(f"DATA PERIODS,{number_of_data_periods},{number_of_records_per_hour},Data,{first_day_year},{df.iloc[0, 1]}/{df.iloc[0, 2]},{df.iloc[-1, 1]}/{df.iloc[-1, 2]}")

    return header_lines

def get_wmo_from_icao_NOAA(icao_code):
    # URL to the NOAA ISD database metadata
    url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
    
    # Fetch the data
    response = requests.get(url)
    
    if response.status_code == 200:
        # Read the CSV data
        csv_data = response.content.decode('utf-8')
        
        # Parse the CSV data
        lines = csv_data.splitlines()
        headers = lines[0].split(',')
        icao_index = headers.index('"ICAO"')
        wmo_index = headers.index('"USAF"')

        for line in lines[1:]:
            fields = line.split(',')
            if fields[icao_index].strip('"') == icao_code.upper():
                return fields[wmo_index].strip('"')
    else:
        print(f"Failed to retrieve data, status code: {response.status_code}")
        return None

def get_time_shift(timezone_name):
    # Get the timezone object
    timezone = pytz.timezone(timezone_name)
    # Get the current time in that timezone
    now = datetime.now(timezone)
    # Retrieve the UTC offset
    utc_offset = now.utcoffset()
    # Convert the offset to hours and minutes
    total_minutes = utc_offset.total_seconds() / 60
    hours = int(total_minutes // 60)
    minutes = int(total_minutes % 60)
    
    return hours

def calculate_hdd_cdd(df, temperature_column):
    """
    Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD) from hourly temperature data in Celsius.

    Parameters:
    df (pd.DataFrame): DataFrame containing hourly temperature data and a datetime column.
    temperature_column (str): Name of the column in the DataFrame containing temperature data in Celsius.

    Returns:
    tuple: Total HDD and CDD as two integers.
    """
    
    # Convert temperature from Celsius to Fahrenheit
    df[temperature_column] = df[temperature_column] * 9 / 5 + 32

    # Set the base temperature for HDD and CDD calculation in Fahrenheit
    base_temperature = 65  # 65°F

    # Calculate daily mean temperature
    df['date'] = df.index.to_series().dt.date  # Convert DatetimeIndex to Series to use .dt accessor
    daily_mean_temp = df.groupby('date')[temperature_column].mean().reset_index()
    daily_mean_temp.columns = ['date', 'mean_temp']  # Rename columns for clarity

    # Calculate Heating Degree Days (HDD)
    daily_mean_temp['HDD'] = (base_temperature - daily_mean_temp['mean_temp']).clip(lower=0)

    # Calculate Cooling Degree Days (CDD)
    daily_mean_temp['CDD'] = (daily_mean_temp['mean_temp'] - base_temperature).clip(lower=0)

    # To get total HDD and CDD for the year, sum the HDD and CDD columns
    total_hdd = daily_mean_temp['HDD'].sum()
    total_cdd = daily_mean_temp['CDD'].sum()

    return int(total_hdd), int(total_cdd)

def run_individual_location(lat, lon, year, file_type, output_path):
    # Run your existing code with these parameters
    data_meteostat_merra2, retrieve_status, info_dict, distance, hdd, cdd = get_noaa_merra2_data(lat, lon, year, file_type)
    #Depending on retrieve_status, we save the file
    if retrieve_status:
        data_meteostat_merra2.to_csv(output_path, header=False, index=False)
        with open(output_path, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_path, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
    return retrieve_status, distance, info_dict['wmo'], hdd, cdd

def get_data_noaa(lat, lon, year):
    #  The time zone used by Meteostat is Coordinated Universal Time (UTC).
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year-1, 12, 31)
    end = datetime(year+1, 1, 2)

    # Use certifi's CA bundle
    # ssl_context = ssl.create_default_context(cafile=certifi.where())
    # Find the closest station
    stations = Stations().nearby(lat, lon) 
    # Get hourly data for the first station
    station_number = 0
    len_data = 0
    while len_data < 8000:
        station_number += 1
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
    timezone = stations.fetch(station_number)['timezone'].values[0]
    #elevation in meters
    elevation = stations.fetch(station_number)['elevation'].values[0]
    #Distance in m
    distance = stations.fetch()['distance'].values[0]
    #WMO
    wmo = str(stations.fetch(station_number).index.values[0])
    #Station Name
    station_name = stations.fetch()['name'].values[0]
    #State
    state = stations.fetch()['region'].values[0]
    #Country
    country = stations.fetch()['country'].values[0]

    return data, timezone, distance, elevation, wmo, station_name, state, country



year = 2022
file_type = 'AMY'
save_folder = 'epws'

# Load the zip codes CSV
zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# for index, row in zipcodes.iterrows():
for index, row in zipcodes.iloc[10000:].iterrows():

    print(index)
    # For specific range, use this line instead:
    # for index, row in zipcodes.iloc[2:3].iterrows():
    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed

    lat = row['lat']
    lon = row['lng']

    output_name = f"{zip_code}_{year}.epw"
    output_path = os.path.join(save_folder, output_name)

    # Check if the file already exists
    if not os.path.exists(output_path):
        retrieve_status, distance, wmo, hdd, cdd = run_individual_location(lat, lon, year, file_type, output_path)
        # Update the DataFrame with the retrieve_status
        zipcodes.at[index, f"Do we have data for {year}?"] = retrieve_status
        zipcodes.at[index, f"distance_location_station_miles_{year}"] = distance * 0.000621371 #Convert from m to miles
        zipcodes.at[index, f"weather_station_wmo_{year}"] = wmo
        zipcodes.at[index, f"hdd_base65F_{year}"] = hdd
        zipcodes.at[index, f"cdd_base65F_{year}"] = cdd

    # Save the DataFrame to the CSV file after each iteration
    zipcodes.to_csv('resources/zip_code_list.csv', index=False)

    # Reopen the file to ensure the latest version is loaded
    zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})


10000


10001


10002


10003


10004


10005


10006


10007


10008
KCCY0


KeyboardInterrupt: 

In [66]:
get_wmo_from_icao_NOAA('KIJD0')

In [54]:
lat = 51.97796
lon = -130.03671
year = 2023

def get_data_noaa(lat, lon, year):
    #  The time zone used by Meteostat is Coordinated Universal Time (UTC).
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year-1, 12, 31)
    end = datetime(year+1, 1, 2)

    # Use certifi's CA bundle
    # ssl_context = ssl.create_default_context(cafile=certifi.where())
    # Find the closest station
    stations = Stations().nearby(lat, lon) 
    # Get hourly data for the first station
    station_number = 0
    len_data = 0
    while len_data < 8000:
        station_number += 1
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)

    print('final_station')
    print(station_number)
    timezone = stations.fetch(station_number)['timezone'].values[0]
    #elevation in meters
    elevation = stations.fetch(station_number)['elevation'].values[0]
    #Distance in m
    distance = stations.fetch()['distance'].values[0]
    print(distance)
    #WMO
    wmo = str(stations.fetch(station_number).index.values[0])
    #Station Name
    station_name = stations.fetch()['name'].values[0]
    #State
    state = stations.fetch()['region'].values[0]
    #Country
    country = stations.fetch()['country'].values[0]

    return data, timezone, distance, elevation, wmo, station_name, state, country

data, timezone, distance, elevation, wmo, station_name, state, country= get_data_noaa(lat, lon, year)

final_station
1
67507.72686757684


In [21]:
data['prcp'].mean()

0.28986264048132593

In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data